<a href="https://colab.research.google.com/github/HashiniTharuka/Building-an-Image-Classifier-with-CNNs/blob/main/MARL-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: Install Packages and Imports
# ============================================================

!pip install torch -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
from datetime import timedelta
import warnings
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
from datetime import datetime

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


The `BedAllocationTrainer` class in your provided context is designed for a DDQN agent, not the MARL agent currently implemented in this notebook. The existing `train_marl` function already handles the training and saving of the best MARL model. If you intend to implement a DDQN agent, the `BedAllocationTrainer` would be relevant then.

In [ ]:
# ============================================================
# 2. MODEL LOADING FUNCTION
# ============================================================

def load_best_model(model_path="best_bed_allocation_model.pkl"):
    """Load the best saved model"""

    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        return None

    with open(model_path, 'rb') as f:
        model_data = pickle.load(f)

    print(f"✅ Model loaded from: {model_path}")
    print(f"   Episode: {model_data['episode']}")
    print(f"   Best Reward: {model_data['best_reward']:.1f}")
    print(f"   Timestamp: {model_data['timestamp']}")

    return model_data

In [ ]:
# ============================================================
# CELL 2: Upload synthetic_500_same_rate.csv
# ============================================================

from google.colab import files

print("="*60)
print("UPLOAD YOUR DATASET")
print("="*60)
print("Please upload: synthetic_500_same_rate.csv")
print("-"*60)

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)
print(f"\n✅ Loaded {len(df)} patients")
print(f"Columns: {df.columns.tolist()}")
print("\nFirst 5 rows:")
print(df.head())

UPLOAD YOUR DATASET
Please upload: synthetic_500_same_rate.csv
------------------------------------------------------------


Saving synthetic_500_same_rate.csv to synthetic_500_same_rate.csv

✅ Loaded 500 patients
Columns: ['patient_id', 'arrival_time', 'triage_level', 'expected_los_hours', 'age', 'gender', 'arrival_hour', 'arrival_day', 'ward']

First 5 rows:
   patient_id                   arrival_time  triage_level  \
0       39798  2026-03-01 08:11:49.496360070             5   
1       39799  2026-03-01 08:53:18.693445111             3   
2       39800  2026-03-01 08:54:48.916206797             5   
3       39801  2026-03-01 10:15:05.167014905             4   
4       39802  2026-03-01 11:43:43.993609301             2   

   expected_los_hours   age gender  arrival_hour arrival_day    ward  
0             137.844  47.2      F             8      Sunday  Ward 3  
1              63.215  39.9      F             8      Sunday  Ward 3  
2              35.076   9.1      F             8      Sunday  Ward 3  
3              17.757  67.8      F            10      Sunday  Ward 3  
4              11.841   7.3      F

In [ ]:
# ============================================================
# CELL 3: Prepare Data for MARL Environment (FIXED)
# ============================================================

# Check and fix column names
if 'triage_level' not in df.columns:
    if 'triage' in df.columns:
        df.rename(columns={'triage': 'triage_level'}, inplace=True)
    elif 'Triage' in df.columns:
        df.rename(columns={'Triage': 'triage_level'}, inplace=True)

if 'expected_los_hours' not in df.columns:
    if 'los_hours' in df.columns:
        df.rename(columns={'los_hours': 'expected_los_hours'}, inplace=True)
    elif 'LOS' in df.columns:
        df.rename(columns={'LOS': 'expected_los_hours'}, inplace=True)  # FIXED: inplace=True

# Create arrival_time if missing OR fix if string
if 'arrival_time' not in df.columns:
    print("Creating synthetic arrival times...")
    base_time = pd.Timestamp('2026-01-01 00:00:00')
    df['arrival_time'] = [
        base_time + timedelta(minutes=i*15 + np.random.randint(0, 60))
        for i in range(len(df))
    ]
else:
    # FIX: Convert to datetime if string
    df['arrival_time'] = pd.to_datetime(df['arrival_time'])

# Ensure triage_level is integer 1-5
df['triage_level'] = df['triage_level'].astype(int).clip(1, 5)

# Ensure LOS is positive
df['expected_los_hours'] = df['expected_los_hours'].abs().clip(0.5, 720)

# Fill any missing values
df = df.dropna(subset=['triage_level', 'expected_los_hours'])

print("\n" + "="*60)
print("DATA READY FOR MARL")
print("="*60)
print(f"Total patients: {len(df)}")
print(f"Triage distribution:")
print(df['triage_level'].value_counts().sort_index())
print(f"\nLOS range: {df['expected_los_hours'].min():.1f} - {df['expected_los_hours'].max():.1f} hours")
print(f"Arrival time range: {df['arrival_time'].min()} to {df['arrival_time'].max()}")

# Sort by arrival time
df = df.sort_values('arrival_time').reset_index(drop=True)

# Verify datetime type
print(f"\n✅ arrival_time type: {df['arrival_time'].dtype}")


DATA READY FOR MARL
Total patients: 500
Triage distribution:
triage_level
1     22
2    173
3     30
4    135
5    140
Name: count, dtype: int64

LOS range: 3.5 - 420.9 hours
Arrival time range: 2026-03-01 08:11:49.496360070 to 2026-04-11 12:12:09.862573896

✅ arrival_time type: datetime64[ns]


In [ ]:
# ============================================================
# CELL 4: MARL Configuration
# ============================================================

BEDS_PER_WARD = 37
STEP_MINUTES = 60
WAIT_PENALTY_K = 0.003
MAX_SIM_DAYS = 70

N_AGENTS = 5  # One per triage level

# Weight grids for actions
WT_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
WW_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_ACTIONS = len(WT_GRID) * len(WW_GRID)  # 64

# Influence weights for triage agents
ALPHA = [0.40, 0.30, 0.15, 0.10, 0.05]

# Training params
EPISODES = 100  # Start with 100 for testing
GAMMA = 0.99
LR = 1e-3
EPSILON_START = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.99
BATCH_SIZE = 64
MEMORY_SIZE = 5000
TARGET_UPDATE_FREQ = 10
TAU = 0.01

In [ ]:
# ============================================================
# CELL 5: MARL Environment (COMPLETE FIX)
# ============================================================

class MARL_Env:
    def __init__(self):
        self.reset()

    def reset(self):
        # Ensure df['arrival_time'] is datetime
        min_time = pd.to_datetime(df['arrival_time'].min())
        self.current_time = min_time - timedelta(minutes=30)
        self.deadline = min_time + timedelta(days=MAX_SIM_DAYS)
        self.queue = deque()
        self.occupied_beds = 0
        self.patients_in_beds = []
        self.patient_counter = 0
        return self._state()

    def _discharge(self):
        self.patients_in_beds = [
            (p, t) for p, t in self.patients_in_beds
            if self.current_time < t
        ]
        self.occupied_beds = len(self.patients_in_beds)

    def _admit(self, pidx):
        los_h = float(df.iloc[pidx]['expected_los_hours'])
        disch = self.current_time + timedelta(hours=los_h)
        self.patients_in_beds.append((pidx, disch))
        self.occupied_beds += 1

    def _state(self):
        self._discharge()
        occ = self.occupied_beds / BEDS_PER_WARD
        q_len = min(len(self.queue) / 30.0, 1.0)

        triage_dist = np.zeros(5)
        longest_wait = 0.0
        if self.queue:
            waits = []
            for p in self.queue:
                t = int(df.iloc[p]['triage_level']) - 1
                triage_dist[t] += 1
                w = (self.current_time - pd.to_datetime(df.iloc[p]['arrival_time'])).total_seconds() / 3600
                waits.append(w)
            triage_dist /= len(self.queue)
            longest_wait = min(max(waits) / 48.0, 1.0)

        return np.array([occ, q_len, *triage_dist, longest_wait, 0., 0.], dtype=np.float32)

    def _fairness(self):
        if len(self.queue) < 2:
            return 0.5
        t_waits = {i: [] for i in range(1, 6)}
        for p in self.queue:
            triage = int(df.iloc[p]['triage_level'])
            wait = (self.current_time - pd.to_datetime(df.iloc[p]['arrival_time'])).total_seconds() / 3600
            t_waits[triage].append(wait)

        avg_w, levels = [], []
        for lv in range(1, 6):
            if t_waits[lv]:
                avg_w.append(np.mean(t_waits[lv]))
                levels.append(lv)
        if len(avg_w) < 2:
            return 0.5
        corr, _ = spearmanr(levels, avg_w)
        return float(np.clip((corr + 1) / 2.0, 0.0, 1.0))

    def step(self, combined_wt, combined_ww):
        self._discharge()

        # Ingest new arrivals
        while (self.patient_counter < len(df) and
               pd.to_datetime(df.iloc[self.patient_counter]['arrival_time']) <= self.current_time):
            self.queue.append(self.patient_counter)
            self.patient_counter += 1

        # Re-order queue
        if len(self.queue) > 1:
            ql = sorted(
                self.queue,
                key=lambda p: (
                    combined_wt * (6 - int(df.iloc[p]['triage_level'])) +
                    combined_ww * (self.current_time - pd.to_datetime(df.iloc[p]['arrival_time'])).total_seconds() / 3600
                ),
                reverse=True
            )
            self.queue = deque(ql)

        # Admit one patient
        admitted_triage = None
        admitted_wait = None
        r_adm = 0.0

        if self.occupied_beds < BEDS_PER_WARD and self.queue:
            pidx = self.queue.popleft()
            triage = int(df.iloc[pidx]['triage_level'])
            admitted_triage = triage
            admitted_wait = (self.current_time - pd.to_datetime(df.iloc[pidx]['arrival_time'])).total_seconds() / 3600.0
            r_adm = (6 - triage) * 2.0
            self._admit(pidx)

        # Metrics
        util = self.occupied_beds / BEDS_PER_WARD
        fairness = self._fairness()
        q_wait = np.mean([(self.current_time - pd.to_datetime(df.iloc[p]['arrival_time'])).total_seconds() / 3600
                         for p in self.queue]) if self.queue else 0.0

        # Reward
        reward = (util * 10.0) + (fairness * 40.0) + r_adm - (q_wait * len(self.queue) * WAIT_PENALTY_K * 0.05) - 0.5

        self.current_time += timedelta(minutes=STEP_MINUTES)
        done = (self.patient_counter >= len(df) and not self.queue) or (self.current_time >= self.deadline)

        return self._state(), reward, done, {
            'fair': fairness,
            'util': util,
            'avg_wait': q_wait,
            'admitted_triage': admitted_triage,
            'admitted_wait': admitted_wait,
        }

In [ ]:
# ============================================================
# CELL 6: Neural Networks
# ============================================================

class AgentDQN(nn.Module):
    def __init__(self, state_dim=10, n_actions=N_ACTIONS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.net(x)


class CentralizedCritic(nn.Module):
    def __init__(self, state_dim=10, n_triage=N_AGENTS):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + n_triage, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, state, triage_onehot):
        x = torch.cat([state, triage_onehot], dim=-1)
        return self.net(x).squeeze(-1)

In [ ]:
# ============================================================
# CELL 7: Replay Buffer (RUN THIS FIRST)
# ============================================================

class ReplayBuffer:
    def __init__(self, capacity=MEMORY_SIZE):
        self.buf = deque(maxlen=capacity)

    def push(self, state, actions, reward, next_state, done, admitted_triage):
        self.buf.append((state, actions, reward, next_state, done, admitted_triage))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        states, actions, rewards, next_states, dones, triages = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(np.array(actions)).to(device),
            torch.FloatTensor(np.array(rewards)).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(np.array(dones)).to(device),
            triages,
        )

    def __len__(self):
        return len(self.buf)

In [ ]:
# ============================================================
# CELL 8: MARL Agent Pool
# ============================================================

class MARLAgentPool:
    def __init__(self):
        self.online = [AgentDQN().to(device) for _ in range(N_AGENTS)]
        self.targets = [AgentDQN().to(device) for _ in range(N_AGENTS)]
        for i in range(N_AGENTS):
            self.targets[i].load_state_dict(self.online[i].state_dict())

        self.critic = CentralizedCritic().to(device)
        self.critic_target = CentralizedCritic().to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())

        self.optimizers = [optim.Adam(self.online[i].parameters(), lr=LR) for i in range(N_AGENTS)]
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=LR)

        self.memory = ReplayBuffer()
        self.epsilon = EPSILON_START
        self.step_count = 0

    def select_actions(self, state):
        state_t = torch.FloatTensor(state[:10]).unsqueeze(0).to(device)
        actions = []
        for i, agent in enumerate(self.online):
            if random.random() < self.epsilon:
                actions.append(random.randint(0, N_ACTIONS - 1))
            else:
                with torch.no_grad():
                    actions.append(agent(state_t).argmax().item())
        return actions

    def negotiate(self, actions):
        combined_wt = 0.0
        combined_ww = 0.0
        for i, a in enumerate(actions):
            wt = WT_GRID[a // len(WW_GRID)]
            ww = WW_GRID[a % len(WW_GRID)]
            combined_wt += ALPHA[i] * wt
            combined_ww += ALPHA[i] * ww
        return combined_wt, combined_ww

    def learn(self):
        if len(self.memory) < BATCH_SIZE:
            return

        states, actions, rewards, next_states, dones, triages = self.memory.sample(BATCH_SIZE)

        # Agent losses
        for i in range(N_AGENTS):
            agent_actions = actions[:, i]
            curr_q = self.online[i](states[:, :10]).gather(1, agent_actions.unsqueeze(1)).squeeze(1)

            with torch.no_grad():
                next_actions = self.online[i](next_states[:, :10]).argmax(dim=1)
                next_q = self.targets[i](next_states[:, :10]).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                target_q = rewards + (1 - dones) * GAMMA * next_q

            loss = nn.MSELoss()(curr_q, target_q)
            self.optimizers[i].zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.online[i].parameters(), 1.0)
            self.optimizers[i].step()

        # Critic loss
        triage_oh = torch.zeros(BATCH_SIZE, N_AGENTS).to(device)
        for b, t in enumerate(triages):
            if t is not None:
                triage_oh[b, int(t) - 1] = 1.0

        critic_val = self.critic(states[:, :10], triage_oh)

        with torch.no_grad():
            next_triage_oh = torch.zeros(BATCH_SIZE, N_AGENTS).to(device)
            target_critic_val = self.critic_target(next_states[:, :10], next_triage_oh)
            critic_target_val = rewards + (1 - dones) * GAMMA * target_critic_val

        critic_loss = nn.MSELoss()(critic_val, critic_target_val)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_opt.step()

        # Soft target updates
        self.step_count += 1
        if self.step_count % TARGET_UPDATE_FREQ == 0:
            for i in range(N_AGENTS):
                for tp, op in zip(self.targets[i].parameters(), self.online[i].parameters()):
                    tp.data.copy_(TAU * op.data + (1.0 - TAU) * tp.data)
            for tp, op in zip(self.critic_target.parameters(), self.critic.parameters()):
                tp.data.copy_(TAU * op.data + (1.0 - TAU) * tp.data)

    def decay_epsilon(self):
        self.epsilon = max(EPSILON_MIN, self.epsilon * EPSILON_DECAY)

In [ ]:
import torch

# ============================================================
# CELL 9: Train MARL
# ============================================================

def train_marl():
    env = MARL_Env()
    pool = MARLAgentPool()

    best_fairness = -1.0
    best_path = "best_marl.pth"

    episode_rewards, episode_fairness, episode_util, episode_waits = [], [], [], []

    print("="*60)
    print("  MARL TRAINING")
    print("="*60)
    print(f"Episodes: {EPISODES}, Patients: {len(df)}")
    print("-"*60)

    for ep in range(EPISODES):
        state = env.reset()
        done = False
        ep_r, ep_f, ep_u, ep_w, steps = 0.0, 0.0, 0.0, 0.0, 0

        while not done:
            actions = pool.select_actions(state)
            combined_wt, combined_ww = pool.negotiate(actions)
            next_state, reward, done, info = env.step(combined_wt, combined_ww)

            pool.memory.push(state, actions, reward, next_state, done, info['admitted_triage'])
            pool.learn()

            state = next_state
            ep_r += reward
            ep_f += info['fair']
            ep_u += info['util']
            ep_w += info['avg_wait']
            steps += 1

        pool.decay_epsilon()

        avg_r = ep_r / steps if steps > 0 else 0
        avg_f = ep_f / steps if steps > 0 else 0
        avg_u = ep_u / steps if steps > 0 else 0
        avg_w = ep_w / steps if steps > 0 else 0

        episode_rewards.append(avg_r)
        episode_fairness.append(avg_f)
        episode_util.append(avg_u)
        episode_waits.append(avg_w)

        is_new_best = False
        if avg_f > best_fairness:
            best_fairness = avg_f
            torch.save({f'agent_{i}': pool.online[i].state_dict() for i in range(N_AGENTS)}, best_path)
            is_new_best = True

        if ep % 10 == 0 or ep == EPISODES - 1 or is_new_best:
            best_marker = " ← 🏆 NEW BEST (saved)" if is_new_best else ""
            print(f"Ep {ep:3d} | Reward: {avg_r:6.2f} | Util: {avg_u*100:5.1f}% | Fair: {avg_f:.4f} | Wait: {avg_w:5.2f}h | ε: {pool.epsilon:.3f}{best_marker}")

    print(f"\n✅ Best Fairness: {best_fairness:.4f}")
    return pool, episode_rewards, episode_fairness, episode_util, episode_waits

pool, rewards, fairness, util, waits = train_marl()

  MARL TRAINING
Episodes: 100, Patients: 500
------------------------------------------------------------
Ep   0 | Reward:  34.21 | Util:  85.7% | Fair: 0.5886 | Wait:  6.52h | ε: 0.990 ← 🏆 NEW BEST (saved)
Ep   1 | Reward:  34.35 | Util:  85.8% | Fair: 0.5922 | Wait:  6.53h | ε: 0.980 ← 🏆 NEW BEST (saved)
Ep   3 | Reward:  34.67 | Util:  85.8% | Fair: 0.6001 | Wait:  6.52h | ε: 0.961 ← 🏆 NEW BEST (saved)
Ep  10 | Reward:  34.52 | Util:  85.7% | Fair: 0.5964 | Wait:  6.56h | ε: 0.895
Ep  18 | Reward:  34.84 | Util:  85.8% | Fair: 0.6042 | Wait:  6.48h | ε: 0.826 ← 🏆 NEW BEST (saved)
Ep  20 | Reward:  34.51 | Util:  85.7% | Fair: 0.5961 | Wait:  6.56h | ε: 0.810
Ep  30 | Reward:  34.26 | Util:  85.7% | Fair: 0.5898 | Wait:  6.53h | ε: 0.732
Ep  40 | Reward:  34.61 | Util:  85.8% | Fair: 0.5984 | Wait:  6.44h | ε: 0.662
Ep  42 | Reward:  35.01 | Util:  85.8% | Fair: 0.6087 | Wait:  6.55h | ε: 0.649 ← 🏆 NEW BEST (saved)
Ep  50 | Reward:  34.36 | Util:  85.7% | Fair: 0.5925 | Wait:  6.56h 

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from collections import deque
from datetime import datetime
import pickle
import os
from scipy.stats import spearmanr
import torch
import torch.nn as nn

# Re-define N_AGENTS and N_ACTIONS for this cell's scope to prevent NameErrors
N_AGENTS = 5
WT_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
WW_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_ACTIONS = len(WT_GRID) * len(WW_GRID)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# ============================================
# 1. MARL ENVIRONMENT (Required for evaluation)
# ============================================

class MARL_BedAllocationEnv:
    """
    Multi-Agent Environment for Bed Allocation
    """
    def __init__(self, df, num_wards=5, beds_per_ward=20):
        self.df = df
        self.num_wards = num_wards
        self.beds_per_ward = beds_per_ward
        self.total_beds = num_wards * beds_per_ward

        self.n_actions = 5
        self.w_t_list = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        self.w_w_list = [0.0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
        self.STEP_MINUTES = 30
        self.WAIT_PENALTY_K = 0.5

        self.reset()

    def reset(self):
        self.current_time = self.df['arrival_time'].min() - pd.Timedelta(minutes=30)
        self.deadline = self.df['arrival_time'].min() + pd.Timedelta(days=70)

        self.ward_queues = [deque() for _ in range(self.num_wards)]
        self.ward_occupied = [0 for _ in range(self.num_wards)]
        self.ward_patients = [[] for _ in range(self.num_wards)]

        self.patient_counter = 0
        self.global_queue = deque()
        self.triage_wait_times = {i: [] for i in range(1, 6)}

        return self._get_state()

    def _discharge(self, ward_idx):
        self.ward_patients[ward_idx] = [
            (p, t) for p, t in self.ward_patients[ward_idx]
            if self.current_time < t
        ]
        self.ward_occupied[ward_idx] = len(self.ward_patients[ward_idx])

    def _admit(self, ward_idx, patient_idx):
        los_h = float(self.df.iloc[patient_idx]['expected_los_hours'])
        discharge_time = self.current_time + pd.Timedelta(hours=los_h)
        self.ward_patients[ward_idx].append((patient_idx, discharge_time))
        self.ward_occupied[ward_idx] += 1

    def _get_state(self):
        states = []
        for w in range(self.num_wards):
            self._discharge(w)

            occupancy = self.ward_occupied[w] / self.beds_per_ward
            queue_len = min(len(self.ward_queues[w]) / 10.0, 1.0)

            triage_dist = np.zeros(5)
            if self.ward_queues[w]:
                for p in self.ward_queues[w]:
                    t = int(self.df.iloc[p]['triage_level']) - 1
                    triage_dist[t] += 1
                triage_dist /= len(self.ward_queues[w])

            longest_wait = 0.0
            if self.ward_queues[w]:
                waits = []
                for p in self.ward_queues[w]:
                    w_time = (self.current_time - self.df.iloc[p]['arrival_time']).total_seconds() / 3600
                    waits.append(w_time)
                longest_wait = min(max(waits) / 48.0, 1.0)

            global_pressure = min(len(self.global_queue) / 30.0, 1.0)

            state = np.array([
                occupancy, queue_len, *triage_dist, longest_wait, global_pressure,
                w / self.num_wards
            ], dtype=np.float32)

            states.append(state)

        return np.array(states)

    def _calculate_fairness(self):
        all_waits = []
        for w in range(self.num_wards):
            for p in self.ward_queues[w]:
                wait = (self.current_time - self.df.iloc[p]['arrival_time']).total_seconds() / 3600
                triage = int(self.df.iloc[p]['triage_level'])
                all_waits.append((triage, wait))

        if len(all_waits) < 2:
            return 0.5

        triage_waits = {i: [] for i in range(1, 6)}
        for triage, wait in all_waits:
            triage_waits[triage].append(wait)

        avg_waits = []
        levels = []
        for lv in range(1, 6):
            if triage_waits[lv]:
                avg_waits.append(np.mean(triage_waits[lv]))
                levels.append(lv)

        if len(avg_waits) < 2:
            return 0.5

        corr, _ = spearmanr(levels, avg_waits)
        return float(np.clip((corr + 1) / 2.0, 0.0, 1.0))

    def step(self, actions):
        for w in range(self.num_wards):
            self._discharge(w)

        while (self.patient_counter < len(self.df) and
               self.df.iloc[self.patient_counter]['arrival_time'] <= self.current_time):
            self.global_queue.append(self.patient_counter)
            self.patient_counter += 1

        for w in range(self.num_wards):
            if self.ward_occupied[w] < self.beds_per_ward and self.global_queue:
                patient = self.global_queue.popleft()
                self.ward_queues[w].append(patient)

        total_reward = 0.0
        admitted_info = []

        for w, action in enumerate(actions):
            self._discharge(w)

            w_t = self.w_t_list[action // len(self.w_w_list)]
            w_w = self.w_w_list[action % len(self.w_w_list)]

            if len(self.ward_queues[w]) > 1:
                ql = sorted(self.ward_queues[w], key=lambda p: (
                    w_t * (6 - int(self.df.iloc[p]['triage_level'])) +
                    w_w * (self.current_time - self.df.iloc[p]['arrival_time']).total_seconds() / 3600
                ), reverse=True)
                self.ward_queues[w] = deque(ql)

            if self.ward_occupied[w] < self.beds_per_ward and self.ward_queues[w]:
                patient = self.ward_queues[w].popleft()
                triage = int(self.df.iloc[patient]['triage_level'])
                admitted_wait = (self.current_time - self.df.iloc[patient]['arrival_time']).total_seconds() / 3600.0

                self._admit(w, patient)
                self.triage_wait_times[triage].append(admitted_wait)

                reward_admit = (6 - triage) * 2.0
                total_reward += reward_admit
                admitted_info.append((w, triage, admitted_wait, reward_admit))

        fairness = self._calculate_fairness()
        total_occupancy = sum(self.ward_occupied) / self.total_beds

        total_wait = 0.0
        total_queue_len = 0
        for w in range(self.num_wards):
            for p in self.ward_queues[w]:
                wait = (self.current_time - self.df.iloc[p]['arrival_time']).total_seconds() / 3600
                total_wait += wait
                total_queue_len += 1

        wait_penalty = (total_wait / max(total_queue_len, 1)) * total_queue_len * self.WAIT_PENALTY_K if total_queue_len > 0 else 0

        reward = (total_occupancy * 10.0) + (fairness * 40.0) - wait_penalty + (len(admitted_info) * 2.0)

        self.current_time += pd.Timedelta(minutes=self.STEP_MINUTES)

        done = (self.patient_counter >= len(self.df) and
                not self.global_queue and
                all(len(q) == 0 for q in self.ward_queues)) or (self.current_time >= self.deadline)

        return self._get_state(), reward, done, {
            'fairness': fairness,
            'occupancy': total_occupancy,
            'admitted': admitted_info,
            'wait_penalty': wait_penalty
        }

# ============================================
# 2. MARL AGENT
# ============================================

class MARL_Agent:
    def __init__(self, state_dim=10, action_dim=N_ACTIONS):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.q_network = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim)
        ).to(self.device)

        self.epsilon = 0.0

    def act(self, state, eval_mode=True):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.q_network(state)
        return q_values.argmax().item()

# ============================================
# 3. SIMULATION FUNCTIONS
# ============================================

def simulate_marl_policy(agents, env):
    """Simulate MARL policy"""
    all_ward_states = env.reset() # This will be (num_wards, 10)
    done = False
    all_fairness, all_occupancy = [], []
    total_reward = 0.0
    triage_wait_times = {i: [] for i in range(1, 6)}

    while not done:
        actions = []
        for w_idx, agent in enumerate(agents): # Iterate through agents and get their respective ward state
            ward_state = all_ward_states[w_idx] # Pass individual ward's state
            action = agent.act(ward_state, eval_mode=True)
            actions.append(action)

        next_all_ward_states, reward, done, info = env.step(actions)

        total_reward += reward
        all_fairness.append(info['fairness'])
        all_occupancy.append(info['occupancy'])

        for _, triage, wait, _ in info['admitted']:
            triage_wait_times[triage].append(wait)

        all_ward_states = next_all_ward_states # Update for next iteration

    return {
        'total_reward': total_reward,
        'avg_fairness': np.mean(all_fairness),
        'avg_utilization': np.mean(all_occupancy),
        'steps': len(all_fairness),
        'triage_wait_times': triage_wait_times
    }

def simulate_fcfs_marl(env):
    """Simulate FCFS policy in MARL environment"""
    env.reset()
    done = False
    all_fairness, all_occupancy = [], []
    total_reward = 0.0
    triage_wait_times = {i: [] for i in range(1, 6)}

    while not done:
        for w in range(env.num_wards):
            if len(env.ward_queues[w]) > 1:
                ql = sorted(env.ward_queues[w], key=lambda p: env.df.iloc[p]['arrival_time'])
                env.ward_queues[w] = deque(ql)

        next_state, reward, done, info = env.step([0] * env.num_wards)

        total_reward += reward
        all_fairness.append(info['fairness'])
        all_occupancy.append(info['occupancy'])

        for _, triage, wait, _ in info['admitted']:
            triage_wait_times[triage].append(wait)

    return {
        'total_reward': total_reward,
        'avg_fairness': np.mean(all_fairness),
        'avg_utilization': np.mean(all_occupancy),
        'steps': len(all_fairness),
        'triage_wait_times': triage_wait_times
    }

# ============================================
# 4. LOAD MODELS
# ============================================

def load_marl_models(save_dir='marl_models', num_agents=N_AGENTS):
    """Load trained MARL models from .pkl files"""
    agents = []
    for i in range(num_agents):
        agent = MARL_Agent(state_dim=10, action_dim=N_ACTIONS)
        model_path = os.path.join(save_dir, f'agent_{i}_best.pkl')

        if os.path.exists(model_path):
            with open(model_path, 'rb') as f:
                model_data = pickle.load(f)
            agent.q_network.load_state_dict(model_data['q_network_state_dict'])
            agent.epsilon = 0.0
            agents.append(agent)
            print(f"✅ Loaded agent {i} from {model_path}")
        else:
            print(f"⚠️ Model not found for agent {i}")
            # Initialize an untrained agent if model not found, useful for initial runs
            agents.append(agent)

    return agents

# ============================================
# 5. PRINT RESULTS (MATCHING DDQN OUTPUT FORMAT)
# ============================================

def print_marl_results(marl_results, fcfs_results):
    """Print MARL vs FCFS comparison in same format as DDQN"""

    # Calculate overall average wait times
    marl_overall_wait = np.mean([item for sublist in marl_results['triage_wait_times'].values()
                                 for item in sublist]) if any(marl_results['triage_wait_times'].values()) else 0.0
    fcfs_overall_wait = np.mean([item for sublist in fcfs_results['triage_wait_times'].values()
                                 for item in sublist]) if any(fcfs_results['triage_wait_times'].values()) else 0.0

    print("\n" + "="*70)
    print("                    MARL vs FCFS COMPARISON")
    print("="*70)
    print(f"{'Metric':<20} {'MARL':>12} {'FCFS':>12} {'MARL Advantage':>15}")
    print("-"*70)

    print(f"{'Avg Reward / step':<20} {marl_results['total_reward']/marl_results['steps']:12.2f} "
          f"{fcfs_results['total_reward']/fcfs_results['steps']:12.2f} "
          f"{(marl_results['total_reward']-fcfs_results['total_reward'])/marl_results['steps']:15.2f}")

    print(f"{'Avg Fairness':<20} {marl_results['avg_fairness']:12.4f} "
          f"{fcfs_results['avg_fairness']:12.4f} "
          f"{marl_results['avg_fairness']-fcfs_results['avg_fairness']:15.4f}")

    print(f"{'Avg Utilization (%)':<20} {marl_results['avg_utilization']*100:11.1f} "
          f"{fcfs_results['avg_utilization']*100:11.1f} "
          f"{(marl_results['avg_utilization']-fcfs_results['avg_utilization'])*100:15.1f}")
    print("="*70)

    print("\n--- Average Wait Time by Triage Level (hours) ---")
    print(f"{'Triage Level':<12} {'MARL':>10} {'FCFS':>10} {'Difference':>12}")
    print("-" * 50)

    for t in range(1, 6):
        marl_avg = np.mean(marl_results['triage_wait_times'][t]) if marl_results['triage_wait_times'][t] else 0.0
        fcfs_avg = np.mean(fcfs_results['triage_wait_times'][t]) if fcfs_results['triage_wait_times'][t] else 0.0
        diff = marl_avg - fcfs_avg
        print(f"{t:<12} {marl_avg:10.2f} {fcfs_avg:10.2f} {diff:12.2f}")

    print("\n" + "="*70) # Closing parenthesis added

